In [1]:
# C1 - Imports et chargement du dataset

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("df_final.csv", parse_dates=["Date"])

print(f"Dimensions : {df.shape}")
print(f"Période    : {df['Date'].min().date()} à {df['Date'].max().date()}")
print(f"Colonnes   : {df.columns.tolist()}")

Dimensions : (408436, 30)
Période    : 2010-03-05 à 2012-10-26
Colonnes   : ['Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday', 'Size', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment', 'Week', 'Month', 'Year', 'Lag_1', 'Lag_2', 'Lag_4', 'Rolling_Mean_4', 'Rolling_Std_4', 'Black_Friday', 'Thanksgiving', 'Xmas_Week', 'Is_Promo', 'Type_A', 'Type_B', 'Type_C']


In [2]:
# C2 - Fonction WMAE personnalisée

def wmae(y_true, y_pred, thanksgiving, black_friday, xmas_week):
    weights = pd.Series(1.0, index=y_true.index)
    weights[(thanksgiving == 1) | (black_friday == 1) | (xmas_week == 1)] = 5
    return round((weights * (y_true - y_pred).abs()).sum() / weights.sum(), 2)

- Métrique officielle Kaggle adaptée au projet
- Pondération x5 sur les semaines de vrais pics de ventes :
  Thanksgiving, Black_Friday, Xmas_Week
- IsHoliday retiré de la pondération — capte SuperBowl, LaborDay, NewYear
  qui n'ont aucun impact réel sur les ventes (confirmé en A5)
- Semaines normales : poids = 1
- Semaines de fêtes : poids = 5

In [3]:
# C3 - Définition des features, cible et validation

df = df.sort_values("Date").reset_index(drop=True)

features = [col for col in df.columns if col not in ["Date", "Weekly_Sales"]]
target   = "Weekly_Sales"

X            = df[features]
y            = df[target]
thanksgiving = df["Thanksgiving"]
black_friday = df["Black_Friday"]
xmas_week    = df["Xmas_Week"]

tscv = TimeSeriesSplit(n_splits=5)

print(f"Nombre de features : {len(features)}")
print(f"Lignes             : {len(X):,}")
print(f"Features           : {features}")

Nombre de features : 28
Lignes             : 408,436
Features           : ['Store', 'Dept', 'IsHoliday', 'Size', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment', 'Week', 'Month', 'Year', 'Lag_1', 'Lag_2', 'Lag_4', 'Rolling_Mean_4', 'Rolling_Std_4', 'Black_Friday', 'Thanksgiving', 'Xmas_Week', 'Is_Promo', 'Type_A', 'Type_B', 'Type_C']


- 28 features retenues - Date et Weekly_Sales exclues
- Thanksgiving, Black_Friday, Xmas_Week extraites séparément pour la fonction WMAE
- TimeSeriesSplit - 5 folds - respecte l'ordre chronologique
- Cible : Weekly_Sales

In [4]:
# C4 - Target Encoding — fonction réutilisable

def target_encoding(X_train, X_val, y_train):
    
    # Clé Store+Dept
    X_train["Store_Dept_key"] = X_train["Store"].astype(str) + "_" + X_train["Dept"].astype(str)
    X_val["Store_Dept_key"]   = X_val["Store"].astype(str)   + "_" + X_val["Dept"].astype(str)

    # Moyennes sur le train uniquement
    store_mean      = y_train.groupby(X_train["Store"].values).mean()
    dept_mean       = y_train.groupby(X_train["Dept"].values).mean()
    store_dept_mean = y_train.groupby(X_train["Store_Dept_key"].values).mean()

    # Ajout au train
    X_train["Store_enc"]      = X_train["Store"].map(store_mean)
    X_train["Dept_enc"]       = X_train["Dept"].map(dept_mean)
    X_train["Store_Dept_enc"] = X_train["Store_Dept_key"].map(store_dept_mean)

    # Ajout à la validation
    X_val["Store_enc"]      = X_val["Store"].map(store_mean)
    X_val["Dept_enc"]       = X_val["Dept"].map(dept_mean)
    X_val["Store_Dept_enc"] = X_val["Store_Dept_key"].map(store_dept_mean)

    # Sécurité — combinaisons absentes du train
    global_mean = y_train.mean()
    X_val["Store_enc"]      = X_val["Store_enc"].fillna(global_mean)
    X_val["Dept_enc"]       = X_val["Dept_enc"].fillna(global_mean)
    X_val["Store_Dept_enc"] = X_val["Store_Dept_enc"].fillna(global_mean)

    # Suppression clé temporaire
    X_train = X_train.drop(columns=["Store_Dept_key"])
    X_val   = X_val.drop(columns=["Store_Dept_key"])

    return X_train, X_val

- Calcule les moyennes de ventes par Store, Dept et Store+Dept
- Moyennes calculées sur le fold train uniquement — pas de data leakage
- Appliquées ensuite sur la validation
- Sécurité : combinaisons absentes du train remplacées par la moyenne globale

In [5]:
# C5 - Random Forest V1 : Weekly_Sales brut
# Boucle CV - Random Forest V1 : Weekly_Sales brut
# Weekly_Sales conservé tel quel — valeurs négatives incluses
# Target Encoding appliqué à chaque fold via la fonction C4
# Pondération WMAE : Thanksgiving, Black_Friday, Xmas_Week (x5)
# n_estimators=100, random_state=42

resultats_v1 = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(X), 1):

    X_train = X.iloc[train_idx].copy()
    X_val   = X.iloc[val_idx].copy()
    y_train = y.iloc[train_idx]
    y_val   = y.iloc[val_idx]

    # Variables WMAE
    t_val  = thanksgiving.iloc[val_idx]
    bf_val = black_friday.iloc[val_idx]
    xw_val = xmas_week.iloc[val_idx]

    # Target Encoding
    X_train, X_val = target_encoding(X_train, X_val, y_train)

    # Modèle
    model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    y_pred   = model.predict(X_val)
    y_pred_s = pd.Series(y_pred, index=y_val.index)

    # Métriques
    resultats_v1.append({
        "Variante" : "V1 - Brut + Target Encoding",
        "Fold"     : f"Fold {fold}",
        "WMAE"     : wmae(y_val, y_pred_s, t_val, bf_val, xw_val),
        "MAE"      : round(mean_absolute_error(y_val, y_pred), 2),
        "RMSE"     : round(np.sqrt(mean_squared_error(y_val, y_pred)), 2),
        "R2"       : round(r2_score(y_val, y_pred), 4),
    })

    print(f"Fold {fold} - WMAE: {resultats_v1[-1]['WMAE']:,.0f}$ "
          f"MAE: {resultats_v1[-1]['MAE']:,.0f}$ "
          f"RMSE: {resultats_v1[-1]['RMSE']:,.0f}$ "
          f"R²: {resultats_v1[-1]['R2']:.4f}")

Fold 1 - WMAE: 4,288$ MAE: 2,878$ RMSE: 9,838$ R²: 0.8348
Fold 2 - WMAE: 1,525$ MAE: 1,525$ RMSE: 3,311$ R²: 0.9765
Fold 3 - WMAE: 2,568$ MAE: 1,757$ RMSE: 6,551$ R²: 0.9213
Fold 4 - WMAE: 1,893$ MAE: 1,733$ RMSE: 3,690$ R²: 0.9748
Fold 5 - WMAE: 1,364$ MAE: 1,364$ RMSE: 2,822$ R²: 0.9836


In [6]:
# C6 - Résultats et export V1

df_v1_results = pd.DataFrame(resultats_v1)

# Moyennes
print("--- Moyennes V1 ---")
print(df_v1_results[["WMAE","MAE","RMSE","R2"]].mean().round(2))



# Export
df_v1_results.to_csv("results_v1.csv", index=False)
print("\nExport results_v1.csv réussi")

--- Moyennes V1 ---
WMAE    2327.61
MAE     1851.45
RMSE    5242.34
R2         0.94
dtype: float64

Export results_v1.csv réussi
